# ML-09 — Validation and Research Claim Audit

This notebook attacks the model before trusting it. It uses the approved gated warehouse when the local Hugging Face token is available, and falls back to the bundled anonymized starter slice when it is not.

## 1. Two paper findings + my methodology questions

**Finding 1 — growing versus declining content (paper, Finding #1, p. 5):** the paper reports that the growing cohort averages about 3.2K words and 184 days of age, while the declining cohort averages about 2.3K words and 230 days. The comparison is useful as an observed portfolio association, but its label comes from a 30-day impression change versus the prior 30 days. My methodology questions are: (a) how much could seasonality or a portfolio mix shift explain the cohort gap, and (b) do the results survive a grouped or time-aware split rather than treating rows as independent?

**Finding 2 — age × freshness matrix (paper, Finding #8, p. 13):** the paper reports that old content refreshed recently has a health score close to the young-and-fresh quadrant, while the 365+ days old and 361+ days untouched cell is a small survivor sample. My methodology questions are: (a) were refreshed pages selected because they already had strategic value or momentum, and (b) is there a pre-refresh baseline or matched comparison that would separate selection from a refresh effect? I therefore treat this as an observed comparison, not causal proof.

In [1]:
from pathlib import Path
import sys
import pandas as pd

repo_candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/FlyRank-ML'), Path('/content/flyrank-ml')]
repo_root = next((p for p in repo_candidates if (p / 'work' / 'ml_track.py').exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from work.ml_track import (
    TARGET,
    ensure_dirs,
    load_analysis_frame,
    make_feature_matrix,
    run_artifacts,
    run_validation,
    write_json,
    write_paper_page,
)

ensure_dirs()
frame = load_analysis_frame()
print(f"Loaded {len(frame):,} rows across {frame['client_id'].nunique():,} client groups")
print(f"Observed snapshot-proxy base rate: {frame[TARGET].mean():.3f}")

random_result = run_validation(frame, 'random')
grouped_result = run_validation(frame, 'grouped_client')

comparison = pd.DataFrame([
    {'split': 'Random row holdout', 'approach': 'Baseline', **random_result['baseline']},
    {'split': 'Random row holdout', 'approach': 'Model', **random_result['model']},
    {'split': 'Grouped client holdout', 'approach': 'Baseline', **grouped_result['baseline']},
    {'split': 'Grouped client holdout', 'approach': 'Model', **grouped_result['model']},
])
display(comparison[['split', 'approach', 'base_rate', 'precision_at_20', 'precision_at_50', 'average_precision', 'roc_auc']].round(3))
print('Primary estimate: grouped client holdout, because client_id repeats across rows.')

C:\Khalil\FlyRank-ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 111,133 rows across 49 client groups
Observed snapshot-proxy base rate: 0.644


,split,approach,base_rate,precision_at_20,precision_at_50,average_precision,roc_auc
0,Random row holdout,Baseline,0.644,0.90,0.82,0.724,0.613
1,Random row holdout,Model,0.644,1.00,1.00,0.847,0.770
2,Grouped client holdout,Baseline,0.598,0.65,0.58,0.710,0.651
3,Grouped client holdout,Model,0.598,0.50,0.58,0.716,0.696


Primary estimate: grouped client holdout, because client_id repeats across rows.


## 2. My model under an honest split (before/after)

The random row holdout is the “before” comparison from the earlier notebook. The grouped client holdout is the “after” stress test: entire pseudonymous client groups are held out, so the model cannot rely on client-specific repetition. Precision@K is shown beside the test base rate, and the same transparent baseline is scored on the same rows.

In [2]:
validation_receipt = {
    'source': frame.attrs.get('source_label'),
    'rows': int(len(frame)),
    'clients': int(frame['client_id'].nunique()),
    'random_split': random_result,
    'grouped_split': grouped_result,
    'primary_split': 'grouped_client',
}
write_json(repo_root / 'work' / 'outputs' / 'ml09_validation_results.json', validation_receipt)
print('Wrote work/outputs/ml09_validation_results.json')

Wrote work/outputs/ml09_validation_results.json


## 3. Leakage audit

The label is derived from `trend_direction`, which itself is derived from `trend_pct`. Neither field is allowed in the model matrix. `content_id` and `client_id` are identifiers for grouping and tracing only; they are not features. The deliberately leaky check below shows why `trend_pct` must stay excluded.

In [3]:
import numpy as np
from sklearn.metrics import roc_auc_score

features, feature_names = make_feature_matrix(frame)
forbidden = {'trend_pct', 'trend_direction', 'content_id', 'client_id'}
leaked_feature_names = sorted(forbidden.intersection(feature_names))
assert not leaked_feature_names, f'Forbidden feature(s) found: {leaked_feature_names}'

trend_pct = frame['trend_pct'].fillna(0).astype(float)
leaky_auc = roc_auc_score(frame[TARGET], -trend_pct) if frame[TARGET].nunique() == 2 else 0.5
print(f'Allowed feature columns: {len(feature_names)}')
print(f'Forbidden feature columns present: {leaked_feature_names}')
print(f'Deliberately leaky trend_pct AUC (diagnostic only, never used): {leaky_auc:.3f}')
print('VERDICT: exclude trend_pct and trend_direction; keep IDs for grouping only.')

Allowed feature columns: 76
Forbidden feature columns present: []
Deliberately leaky trend_pct AUC (diagnostic only, never used): 1.000
VERDICT: exclude trend_pct and trend_direction; keep IDs for grouping only.


## 4. Claim rewrite

**Too strong:** “The model predicts which pages Google will demote and proves that refreshing them will restore traffic.”

**Evidence-safe rewrite:** “In this anonymized snapshot, the model ranked pages associated with the declining-trend proxy above a transparent review rule on a grouped client holdout. The queue is directional decision support for human review; it does not establish causes, future Google behavior, or the effect of a refresh.”

In [4]:
print('Claim check: observed / measured / directional / decision-support language used.')
print('No causal or algorithm-reconstruction claim is made.')

Claim check: observed / measured / directional / decision-support language used.
No causal or algorithm-reconstruction claim is made.


## Self-check

- [x] Every section above is filled with markdown and executable checks
- [x] Random and grouped results use the same baseline and metrics
- [x] Label-derived fields and IDs are excluded from features
- [x] The notebook writes a JSON receipt in `work/outputs/`
- [x] Claims stay observed, measured, directional, and decision-support oriented